# Ladybug Conversations Explorer

This notebook loads and displays the processed Ladybug conversation data for review and analysis.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# Set pandas display options for better viewing
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)

## Load the Data

In [ ]:
# Load the enhanced cache
cache_path = Path('.progress/ladybug_enhanced_cache.json')

with open(cache_path, 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} conversations")

## Convert to DataFrame

In [ ]:
# Flatten the data for easier viewing
rows = []
for item in data:
    conv_id = item['id']
    messages = item['messages']
    
    # Extract system prompt
    system_prompt = next((m['content'] for m in messages if m['role'] == 'system'), '')
    
    # Extract conversation turns (excluding system)
    turns = [m for m in messages if m['role'] != 'system']
    num_turns = len(turns)
    
    # Format conversation as readable text
    conv_text = '\n'.join([f"{t['role'].upper()}: {t['content']}" for t in turns])
    
    rows.append({
        'id': conv_id,
        'num_turns': num_turns,
        'system_prompt': system_prompt[:100] + '...' if len(system_prompt) > 100 else system_prompt,
        'conversation': conv_text,
        'full_messages': messages
    })

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
df.head()

## Statistics

In [ ]:
print("Turn count distribution:")
print(df['num_turns'].value_counts().sort_index())
print(f"\nAverage turns: {df['num_turns'].mean():.2f}")
print(f"Min turns: {df['num_turns'].min()}")
print(f"Max turns: {df['num_turns'].max()}")

## View Sample Conversations

In [ ]:
def display_conversation(idx):
    """Display a single conversation nicely formatted."""
    row = df.iloc[idx]
    print(f"=" * 80)
    print(f"ID: {row['id']}")
    print(f"Turns: {row['num_turns']}")
    print(f"=" * 80)
    print(f"\nSYSTEM: {row['full_messages'][0]['content']}")
    print(f"\n" + "-" * 40 + "\n")
    for msg in row['full_messages'][1:]:
        role = msg['role'].upper()
        print(f"{role}: {msg['content']}\n")

# Display first 3 conversations
for i in range(min(3, len(df))):
    display_conversation(i)
    print("\n")

## Browse Conversations Interactively

In [ ]:
# Change this index to browse different conversations
CONVERSATION_INDEX = 0

display_conversation(CONVERSATION_INDEX)

## Search Conversations

In [ ]:
def search_conversations(keyword):
    """Search for conversations containing a keyword."""
    mask = df['conversation'].str.contains(keyword, case=False, na=False)
    results = df[mask]
    print(f"Found {len(results)} conversations containing '{keyword}'")
    return results

# Example search
search_results = search_conversations("Cat Noir")
search_results[['id', 'num_turns', 'conversation']].head()

## Check for Issues

In [ ]:
# Check for action descriptions that shouldn't be there
action_patterns = ['*pushes', '*grabs', '*runs', '*walks', '*looks around', '*nervously fidgets']

print("Checking for unwanted action descriptions...\n")
for pattern in action_patterns:
    mask = df['conversation'].str.contains(pattern, case=False, na=False, regex=False)
    count = mask.sum()
    if count > 0:
        print(f"'{pattern}': {count} occurrences")
        # Show first example
        example = df[mask].iloc[0]['conversation']
        print(f"  Example: {example[:200]}...\n")

## View System Prompts Variety

In [ ]:
# Extract unique system prompts
unique_prompts = df['full_messages'].apply(lambda x: x[0]['content']).unique()
print(f"Number of unique system prompts: {len(unique_prompts)}\n")

for i, prompt in enumerate(unique_prompts[:10]):
    print(f"{i+1}. {prompt}\n")

## Export Options

In [ ]:
# Export to CSV for easy viewing in spreadsheet
# df[['id', 'num_turns', 'conversation']].to_csv('ladybug_conversations.csv', index=False)

# Export full data to JSONL (training format)
# with open('ladybug_train.jsonl', 'w') as f:
#     for item in data:
#         f.write(json.dumps({'messages': item['messages']}, ensure_ascii=False) + '\n')